In [1]:
# # 01 - Carga, limpieza y transformación de datos
# Dataset: ventas de videojuegos (vgsales.csv). Ventas en millones de unidades.

# %%
from pathlib import Path
import numpy as np
import pandas as pd

# Los notebooks viven en /notebooks, así que la raíz del repo es la carpeta anterior
RAIZ = Path.cwd().parent
RUTA_DATOS = RAIZ / "data" / "vgsales.csv"

df = pd.read_csv(RUTA_DATOS)

print("Dimensiones:", df.shape)
display(df.head())
df.info()

# %%
# REVISAR NULOS Y DUPLICADOS
nulos = df.isnull().sum()
tabla_nulos = pd.DataFrame({
    "Nulos": nulos,
    "% nulos": (nulos / len(df) * 100).round(2)
})
display(tabla_nulos)

print("Duplicados exactos:", df.duplicated().sum())
print("Filas con mismo Name + Platform:", df.duplicated(subset=["Name", "Platform"]).sum())

# Los de mismo Name + Platform NO se eliminan: tienen años o ventas distintas
# (ej. remakes o regiones capturadas por separado), no son copias exactas.
display(
    df[df.duplicated(subset=["Name", "Platform"], keep=False)]
    .sort_values("Name")
    .head(10)
)

# %%
# LIMPIEZA
df_limpio = df.copy()

# 1) Quitar espacios sobrantes en columnas de texto
for col in ["Name", "Platform", "Genre", "Publisher"]:
    df_limpio[col] = df_limpio[col].astype("string").str.strip()

# 2) Publisher nulo -> "Unknown" (son pocos y el juego sigue siendo útil para ventas)
df_limpio["Publisher"] = df_limpio["Publisher"].fillna("Unknown")

# 3) Year viene como decimal porque tiene nulos -> entero que acepta nulos
#    No se imputa: inventar un año sesgaría el análisis temporal
df_limpio["Year"] = df_limpio["Year"].astype("Int64")

# 4) Eliminar duplicados exactos
df_limpio = df_limpio.drop_duplicates()

print("Filas antes:", len(df), "| después:", len(df_limpio))
print("Juegos sin año (se excluyen solo en el análisis temporal):", df_limpio["Year"].isna().sum())

# %%
# VALIDAR CONSISTENCIA: Global_Sales debería ser la suma de las 4 regiones
columnas_region = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]

suma_regiones = df_limpio[columnas_region].sum(axis=1)
diferencia = (df_limpio["Global_Sales"] - suma_regiones).abs()

print("Filas donde Global_Sales difiere de la suma regional por más de 0.05 M:",
      (diferencia > 0.05).sum())
print("Diferencia máxima:", round(diferencia.max(), 2))

# %%
# NUEVAS COLUMNAS PARA EL ANÁLISIS
df_limpio["Decada"] = (df_limpio["Year"] // 10 * 10).astype("Int64")

# Participación de cada región en las ventas globales de cada juego
for col in columnas_region:
    df_limpio[col.replace("_Sales", "_Share")] = (
        df_limpio[col] / df_limpio["Global_Sales"]
    ).round(4)

display(df_limpio.head())

# %%
# ACOMODO DE DATOS: formato ancho -> formato largo (una fila por juego y región)
# Este formato facilita agrupar y graficar por región
ventas_region = df_limpio.melt(
    id_vars=["Name", "Platform", "Year", "Genre", "Publisher"],
    value_vars=columnas_region,
    var_name="Region",
    value_name="Ventas_millones"
)
ventas_region["Region"] = ventas_region["Region"].str.replace("_Sales", "", regex=False)

display(ventas_region.head())
print("Dimensiones formato largo:", ventas_region.shape)

# %%
# REVISAR CUÁNTOS JUEGOS HAY POR AÑO (importante para saber hasta cuándo hay datos completos)
juegos_por_anio = df_limpio.groupby("Year").size()
display(juegos_por_anio.tail(8))

# Los últimos años tienen muy pocos registros: el dataset está incompleto ahí.
# Esto se tomará en cuenta en el análisis y en el forecasting.

Dimensiones: (16598, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


,Nulos,% nulos
Rank,0,0.00
Name,0,0.00
Platform,0,0.00
Year,271,1.63
Genre,0,0.00
Publisher,58,0.35
NA_Sales,0,0.00
EU_Sales,0,0.00
JP_Sales,0,0.00
Other_Sales,0,0.00


Duplicados exactos: 0
Filas con mismo Name + Platform: 5


...

Filas antes: 16598 | después: 16598
Juegos sin año (se excluyen solo en el análisis temporal): 271
Filas donde Global_Sales difiere de la suma regional por más de 0.05 M: 0
Diferencia máxima: 0.02


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Decada,NA_Share,EU_Share,JP_Share,Other_Share
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,2000,0.5015,0.3507,0.0456,0.1022
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,1980,0.7227,0.0890,0.1692,0.0191
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,2000,0.4425,0.3596,0.1058,0.0924
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,2000,0.4773,0.3336,0.0994,0.0897
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,1990,0.3593,0.2834,0.3258,0.0319


,Name,Platform,Year,Genre,Publisher,Region,Ventas_millones
0,Wii Sports,Wii,2006,Sports,Nintendo,NA,41.49
1,Super Mario Bros.,NES,1985,Platform,Nintendo,NA,29.08
2,Mario Kart Wii,Wii,2008,Racing,Nintendo,NA,15.85
3,Wii Sports Resort,Wii,2009,Sports,Nintendo,NA,15.75
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,NA,11.27


Dimensiones formato largo: (66392, 7)


Year
2011    1139
2012     657
2013     546
2014     582
2015     614
2016     344
2017       3
2020       1
dtype: int64